In [1]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.discrete.count_model import ZeroInflatedPoisson

In [2]:
df = pd.read_csv("hotel_bookings.csv")
df

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.00,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.00,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.00,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.00,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.00,0,1,Check-Out,2015-07-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119385,City Hotel,0,23,2017,August,35,30,2,5,2,...,No Deposit,394.0,NaN,0,Transient,96.14,0,0,Check-Out,2017-09-06
119386,City Hotel,0,102,2017,August,35,31,2,5,3,...,No Deposit,9.0,NaN,0,Transient,225.43,0,2,Check-Out,2017-09-07
119387,City Hotel,0,34,2017,August,35,31,2,5,2,...,No Deposit,9.0,NaN,0,Transient,157.71,0,4,Check-Out,2017-09-07
119388,City Hotel,0,109,2017,August,35,31,2,5,2,...,No Deposit,89.0,NaN,0,Transient,104.40,0,0,Check-Out,2017-09-07


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  object 
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  object 
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal            

In [4]:
df.isnull().sum()

hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               4
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                           0
agent                              16340
company         

In [5]:
df = df[["lead_time", "adr", "previous_cancellations"]]
df

,lead_time,adr,previous_cancellations
0,342,0.00,0
1,737,0.00,0
2,7,75.00,0
3,13,75.00,0
4,14,98.00,0
...,...,...,...
119385,23,96.14,0
119386,102,225.43,0
119387,34,157.71,0
119388,109,104.40,0


In [6]:
df.isnull().sum()

lead_time                 0
adr                       0
previous_cancellations    0
dtype: int64

In [7]:
df.dropna(inplace=True)

C:\Users\tamil\AppData\Local\Temp\ipykernel_20396\1379821321.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.dropna(inplace=True)


In [8]:
X = df[["lead_time", "adr"]]
X

,lead_time,adr
0,342,0.00
1,737,0.00
2,7,75.00
3,13,75.00
4,14,98.00
...,...,...
119385,23,96.14
119386,102,225.43
119387,34,157.71
119388,109,104.40


In [9]:
y = df["previous_cancellations"]
y

0         0
1         0
2         0
3         0
4         0
         ..
119385    0
119386    0
119387    0
119388    0
119389    0
Name: previous_cancellations, Length: 119390, dtype: int64

In [10]:
X = sm.add_constant(X)
X

,const,lead_time,adr
0,1.0,342,0.00
1,1.0,737,0.00
2,1.0,7,75.00
3,1.0,13,75.00
4,1.0,14,98.00
...,...,...,...
119385,1.0,23,96.14
119386,1.0,102,225.43
119387,1.0,34,157.71
119388,1.0,109,104.40


In [11]:
model = ZeroInflatedPoisson(endog=y,exog=X,exog_infl=X)
result = model.fit(maxiter=200,method="bfgs")

C:\Users\tamil\anaconda3\Lib\site-packages\statsmodels\discrete\discrete_model.py:1331: RuntimeWarning: overflow encountered in exp
  return -np.exp(XB) +  endog*XB - gammaln(endog+1)
C:\Users\tamil\anaconda3\Lib\site-packages\statsmodels\discrete\discrete_model.py:1508: RuntimeWarning: overflow encountered in exp
  L = np.exp(np.dot(X,params) + offset + exposure)
C:\Users\tamil\anaconda3\Lib\site-packages\statsmodels\discrete\discrete_model.py:1074: RuntimeWarning: overflow encountered in exp
  return np.exp(linpred)
C:\Users\tamil\anaconda3\Lib\site-packages\statsmodels\discrete\discrete_model.py:1331: RuntimeWarning: overflow encountered in exp
  return -np.exp(XB) +  endog*XB - gammaln(endog+1)


Optimization terminated successfully.
         Current function value: 0.274662
         Iterations: 38
         Function evaluations: 61
         Gradient evaluations: 52


In [12]:
print(result.summary())

                      ZeroInflatedPoisson Regression Results                      
Dep. Variable:     previous_cancellations   No. Observations:               119390
Model:                ZeroInflatedPoisson   Df Residuals:                   119387
Method:                               MLE   Df Model:                            2
Date:                    Mon, 29 Jun 2026   Pseudo R-squ.:                  0.1121
Time:                            11:34:38   Log-Likelihood:                -32792.
converged:                           True   LL-Null:                       -36933.
Covariance Type:                nonrobust   LLR p-value:                     0.000
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
inflate_const         3.1740      0.046     69.750      0.000       3.085       3.263
inflate_lead_time    -0.0100      0.000    -65.158      0.000      -0.010     

In [13]:
print(df["previous_cancellations"].value_counts())

previous_cancellations
0     112906
1       6051
2        116
3         65
24        48
11        35
4         31
26        26
25        25
6         22
5         19
19        19
14        14
13        12
21         1
Name: count, dtype: int64


In [14]:
zero_percentage = (df["previous_cancellations"] == 0).mean() * 100
print("Zero Percentage:", round(zero_percentage, 2), "%")

Zero Percentage: 94.57 %


In [16]:
lead_time = int(input("Enter Lead Time: "))
adr = float(input("Enter ADR: "))
new_data = pd.DataFrame({"const": [1],"lead_time": [lead_time],"adr": [adr]})
prediction = result.predict(exog=new_data,exog_infl=new_data)
print("Predicted Previous Cancellations:", round(prediction[0]))

Enter Lead Time:  180
Enter ADR:  90


Predicted Previous Cancellations: 0
